In the modular approach I will create separate files for different model types. This one will be for Beta Sil App Score

# Define Library

In [1]:
# %% [markdown]
# # Jupyter Notebook Loading Header
#
# This is a custom loading header for Jupyter Notebooks in Visual Studio Code.
# It includes common imports and settings to get you started quickly.
# %% [markdown]
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from google.cloud import storage
import os
import tempfile
import time
from datetime import datetime
import uuid
import joblib
import uuid
from sklearn.metrics import roc_auc_score
from datetime import datetime, timedelta
import gcsfs
import duckdb as dd
import pickle
import joblib
from typing import Union
import io
path = r'C:\Users\Dwaipayan\AppData\Roaming\gcloud\legacy_credentials\dchakroborti@tonikbank.com\adc.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"

# %% [markdown]
## Configure Settings
# Set options or configurations as needed
pd.set_option('display.max_columns', None)
pd.set_option("Display.max_rows", 100)

# Function

## calculate_gini

In [2]:
def calculate_gini(pd_scores, bad_indicators):
    """
    Calculate Gini coefficient from scores and binary indicators
    
    Parameters:
    pd_scores: array-like of scores/probabilities
    bad_indicators: array-like of binary outcomes (0/1)
    
    Returns:
    float: Gini coefficient
    """
    # Convert inputs to numpy arrays and ensure they're numeric
    pd_scores = np.array(pd_scores, dtype=float)
    bad_indicators = np.array(bad_indicators, dtype=int)
    
    # Check for valid input data
    if len(pd_scores) == 0 or len(bad_indicators) == 0:
        return np.nan
    
    # Check if we have both good and bad cases (needed for ROC AUC)
    if len(np.unique(bad_indicators)) < 2:
        return np.nan
    
    # Calculate AUC using sklearn
    try:
        auc = roc_auc_score(bad_indicators, pd_scores)
        # Calculate Gini from AUC
        gini = 2 * auc - 1
        return gini
    except ValueError:
        return np.nan

## calculate_periodic_gini_prod_ver_trench_dimfact

In [3]:
import pandas as pd
import numpy as np
from datetime import timedelta
from itertools import combinations

def calculate_gini(scores, labels):
    """
    Calculate Gini coefficient with proper handling of edge cases
    
    Returns np.nan when:
    - Fewer than 2 observations
    - No positive labels (sum of labels = 0)
    """
    n = len(scores)
    if n < 2:
        return np.nan
    
    label_sum = np.sum(labels)
    
    # Handle case where no positive labels exist (all zeros)
    # This prevents division by zero warning
    if label_sum == 0:
        return np.nan
    
    sorted_indices = np.argsort(scores)
    sorted_labels = labels.iloc[sorted_indices].values
    cumsum_labels = np.cumsum(sorted_labels)
    
    gini = 1 - 2 * np.sum(cumsum_labels) / (n * label_sum)
    return gini


def calculate_periodic_gini_prod_ver_trench_dimfact(df, score_column, label_column, namecolumn, 
                                        data_selection_column=None,
                                        model_version_column=None, trench_column=None, 
                                        loan_type_column=None, loan_product_type_column=None,
                                        ostype_column=None,
                                        risk_segment_column=None,
                                        risk_segment_final_column=None,
                                        account_id_column=None):
    """
    Calculate periodic Gini coefficients and return Power BI-friendly long format
    with fact and dimension tables.
    
    Returns:
    - fact_table: Long format with one row per segment per period
    - dimension_table: Unique segment combinations for filtering
    
    Parameters:
    df: DataFrame with disbursement dates and score/label columns
    score_column: name of the score column
    label_column: name of the label column
    namecolumn: name for the bad rate label
    data_selection_column: (optional) name of column for data selection (Test/Train)
    model_version_column: (optional) name of column for model version
    trench_column: (optional) name of column for trench category
    loan_type_column: (optional) name of loan type column
    loan_product_type_column: (optional) name of loan product type column
    ostype_column: (optional) name of column for OS type
    account_id_column: (optional) name of column for distinct account IDs
    """
    # Input validation
    required_columns = ['disbursementdate', score_column, label_column]
    if not all(col in df.columns for col in required_columns):
        raise ValueError(f"Missing required columns. Need: {required_columns}")
    
    optional_columns = {
        'data_selection': data_selection_column,
        'model_version': model_version_column,
        'trench': trench_column,
        'loan_type': loan_type_column,
        'loan_product_type': loan_product_type_column,
        'ostype': ostype_column,
        'risk_segment': risk_segment_column,
        'risk_segment_final': risk_segment_final_column,
        'account_id': account_id_column
    }
    
    for col_name, col in optional_columns.items():
        if col and col not in df.columns:
            raise ValueError(f"{col_name.replace('_', ' ').title()} column '{col}' not found in dataframe")
    
    # Create a copy to avoid modifying original dataframe
    df = df.copy()
    
    # Ensure date is datetime type
    df['disbursementdate'] = pd.to_datetime(df['disbursementdate'])
    
    # Ensure score and label columns are numeric
    df[score_column] = pd.to_numeric(df[score_column], errors='coerce')
    df[label_column] = pd.to_numeric(df[label_column], errors='coerce')
    
    # Drop rows with invalid values
    df = df.dropna(subset=[score_column, label_column])
    
    # Define list of datasets to process
    datasets_to_process = [('Overall', df, {})]
    
    # Create list of available segment columns
    segment_columns = []
    if data_selection_column:
        segment_columns.append(('DataSelection', data_selection_column))
    if model_version_column:
        segment_columns.append(('ModelVersion', model_version_column))
    if trench_column:
        segment_columns.append(('Trench', trench_column))
    if loan_type_column:
        segment_columns.append(('LoanType', loan_type_column))
    if loan_product_type_column:
        segment_columns.append(('ProductType', loan_product_type_column))
    if ostype_column:
        segment_columns.append(('OSType', ostype_column))
    if risk_segment_column:
        segment_columns.append(('risk_segment', risk_segment_column))
    if risk_segment_final_column:
        segment_columns.append(('risk_segment_final', risk_segment_final_column))
    
    # Generate all possible combinations of segment columns
    for r in range(1, len(segment_columns) + 1):
        for combo in combinations(segment_columns, r):
            def generate_combinations(df, segment_columns, index=0, current_filter=None, current_name=''):
                if current_filter is None:
                    current_filter = {}
                
                if index >= len(segment_columns):
                    filtered_df = df
                    for col, val in current_filter.items():
                        filtered_df = filtered_df[filtered_df[col] == val]
                    
                    if len(filtered_df) > 0:
                        yield (current_name.strip('_'), filtered_df, current_filter.copy())
                    return
                
                seg_name, seg_col = segment_columns[index]
                for seg_value in sorted(df[seg_col].dropna().unique()):
                    new_filter = current_filter.copy()
                    new_filter[seg_col] = seg_value
                    new_name = current_name + f'{seg_name}_{seg_value}_'
                    
                    yield from generate_combinations(df, segment_columns, index + 1, new_filter, new_name)
            
            for combo_name, combo_df, combo_metadata in generate_combinations(df, list(combo)):
                datasets_to_process.append((combo_name, combo_df, combo_metadata))
    
    all_results = []
    
    # Process each dataset
    for dataset_name, dataset_df, metadata in datasets_to_process:
        # Calculate weekly Gini
        dataset_df_copy = dataset_df.copy()
        dataset_df_copy['week'] = dataset_df_copy['disbursementdate'].dt.to_period('W')
        weekly_gini = dataset_df_copy.groupby('week').apply(
            lambda x: calculate_gini(x[score_column], x[label_column])
            if len(x) >= 10 else np.nan
        ).reset_index(name='gini_value')
        weekly_gini['period'] = 'Week'
        weekly_gini['start_date'] = weekly_gini['week'].apply(lambda x: x.to_timestamp())
        weekly_gini['end_date'] = weekly_gini['start_date'] + timedelta(days=6)
        
        # Add distinct account count for weekly
        if account_id_column:
            weekly_account_counts = dataset_df_copy.groupby('week')[account_id_column].nunique().reset_index()
            weekly_account_counts.columns = ['week', 'distinct_accounts']
            weekly_gini = weekly_gini.merge(weekly_account_counts, on='week', how='left')
        else:
            weekly_gini['distinct_accounts'] = None
        
        weekly_gini = weekly_gini[['start_date', 'end_date', 'gini_value', 'period', 'distinct_accounts']]
        
        # Calculate monthly Gini
        dataset_df_copy = dataset_df.copy()
        dataset_df_copy['month'] = dataset_df_copy['disbursementdate'].dt.to_period('M')
        monthly_gini = dataset_df_copy.groupby('month').apply(
            lambda x: calculate_gini(x[score_column], x[label_column])
            if len(x) >= 20 else np.nan
        ).reset_index(name='gini_value')
        monthly_gini['period'] = 'Month'
        monthly_gini['start_date'] = monthly_gini['month'].apply(lambda x: x.to_timestamp())
        monthly_gini['end_date'] = monthly_gini['start_date'] + pd.DateOffset(months=1) - pd.Timedelta(days=1)
        
        # Add distinct account count for monthly
        if account_id_column:
            monthly_account_counts = dataset_df_copy.groupby('month')[account_id_column].nunique().reset_index()
            monthly_account_counts.columns = ['month', 'distinct_accounts']
            monthly_gini = monthly_gini.merge(monthly_account_counts, on='month', how='left')
        else:
            monthly_gini['distinct_accounts'] = None
        
        monthly_gini = monthly_gini[['start_date', 'end_date', 'gini_value', 'period', 'distinct_accounts']]
        
        # Combine results for this dataset
        gini_results = pd.concat([weekly_gini, monthly_gini], ignore_index=True)
        gini_results = gini_results.sort_values(by='start_date').reset_index(drop=True)
        
        # Add metadata columns
        gini_results['Model_Name'] = score_column
        gini_results['bad_rate'] = namecolumn
        gini_results['segment_type'] = dataset_name
        
        # Add individual segment components
        gini_results['data_selection'] = metadata.get(data_selection_column, None) if data_selection_column else None
        gini_results['model_version'] = metadata.get(model_version_column, None) if model_version_column else None
        gini_results['trench_category'] = metadata.get(trench_column, None) if trench_column else None
        gini_results['loan_type'] = metadata.get(loan_type_column, None) if loan_type_column else None
        gini_results['loan_product_type'] = metadata.get(loan_product_type_column, None) if loan_product_type_column else None
        gini_results['ostype'] = metadata.get(ostype_column, None) if ostype_column else None
        gini_results['risk_segment'] = metadata.get(risk_segment_column, None) if risk_segment_column else None
        gini_results['risk_segment_final'] = metadata.get(risk_segment_final_column, None) if risk_segment_final_column else None   
        
        all_results.append(gini_results)
    
    # Combine all results
    fact_table = pd.concat(all_results, ignore_index=True)
    
    # Create dimension table (unique segment combinations for filtering)
    dimension_table = fact_table[['Model_Name', 'bad_rate', 'segment_type', 'data_selection', 'model_version', 
                                   'trench_category', 'loan_type', 'loan_product_type', 'ostype', 'risk_segment', 'risk_segment_final']].drop_duplicates().reset_index(drop=True)
    dimension_table['segment_id'] = range(len(dimension_table))
    
    # Add segment_id to fact table
    fact_table = fact_table.merge(dimension_table[['segment_id', 'Model_Name', 'bad_rate', 'segment_type', 
                                                     'data_selection', 'model_version', 'trench_category', 'loan_type', 'loan_product_type', 'ostype', 'risk_segment', 'risk_segment_final']], 
                                  on=['Model_Name', 'bad_rate', 'segment_type', 'data_selection', 'model_version', 
                                      'trench_category', 'loan_type', 'loan_product_type', 'ostype', 'risk_segment', 'risk_segment_final'], 
                                  how='left')
    
    # Reorder columns in fact table
    fact_table = fact_table[['segment_id', 'start_date', 'end_date', 'period', 'gini_value', 'distinct_accounts',
                             'Model_Name', 'bad_rate', 'segment_type', 'data_selection', 'model_version', 'trench_category', 
                             'loan_type', 'loan_product_type', 'ostype', 'risk_segment', 'risk_segment_final']]
    
    # Reorder columns in dimension table
    dimension_table = dimension_table[['segment_id', 'Model_Name', 'bad_rate', 'segment_type', 'data_selection', 
                                        'model_version', 'trench_category', 'loan_type', 'loan_product_type', 'ostype', 'risk_segment', 'risk_segment_final']]
    
    return fact_table, dimension_table


# Usage:
# fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
#     df_concat, 
#     'Alpha_cic_sil_score', 
#     'deffpd0', 
#     'FPD0',
#     data_selection_column='Data_selection',
#     model_version_column='modelVersionId',
#     trench_column='trenchCategory',
#     loan_type_column='loan_type',
#     loan_product_type_column='loan_product_type',
#     ostype_column='osType',
#     account_id_column='digitalLoanAccountId'
# )
# 
# # In Power BI:
# # 1. Import fact_table and dimension_table
# # 2. Create relationship: dimension_table[segment_id] -> fact_table[segment_id]
# # 3. Use dimension table columns as filters (including ostype)
# # 4. Create DAX measures:
# #    - Gini Measure = AVERAGE(fact_table[gini_value])
# #    - Account Count = SUM(fact_table[distinct_accounts])
# # 5. Use start_date, end_date, period for time-based analysis

# update_tables

In [12]:

def update_tables(fact_table: pd.DataFrame, dimension_table: pd.DataFrame, model_name: str, product: str) -> tuple:
    """
    Updates fact_table and dimension_table:
    - Sets 'Model_display_name' to the given model_name
    - Replaces NaN values in specified columns with 'Overall'
    
    Returns:
        Updated fact_table and dimension_table as a tuple
    """
    # Columns where missing values should be replaced
    cols_to_replace = ['model_version', 'trench_category', 'loan_type', 'loan_product_type', 'ostype','risk_segment', 'risk_segment_final']
    
    # Update fact_table
    fact_table['Model_display_name'] = model_name
    fact_table['Product_Category'] = product
    fact_table[cols_to_replace] = fact_table[cols_to_replace].fillna('Overall')
    
    # Update dimension_table
    dimension_table['Model_display_name'] = model_name
    dimension_table['Product_Category'] = product
    dimension_table[cols_to_replace] = dimension_table[cols_to_replace].fillna('Overall')
    
    return fact_table, dimension_table


## alpha_stack_model_sil_credo_score 

## FPD0 

### Test 

In [5]:
sq = f"""
with modelname as 
  (  SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
  case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
      deviceOs osType,
    FROM prj-prod-dataplatform.audit_balance.ml_model_run_details mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
  -- and modelVersionId = 'v1'
      ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where  modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  coalesce(cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.credo_score')AS FLOAT64), 
  cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.s_credo_score')AS FLOAT64)) AS credo_score,
  calcFeature,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  'Prod' Data_selection,  
  deffpd0,
  flg_mature_fpd0,
  loanmaster.new_loan_type,
  modelVersionId, r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
        coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
     coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  left join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
  left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where 
  loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and flg_mature_fpd0 = 1
  )
  select *
  from base
  where credo_score is not null
 qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
   ;
"""
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()


The shape of the dataframe downloaded is:	 (88063, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,credo_score,calcFeature,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd0,flg_mature_fpd0,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3789565,014b2b18-a79c-4083-937a-8a2e123c3068,60837895650013,alpha_stack_model_sil,0.1300,"{""sb_demo_score"": 0.0386566044, ""s_cic_score"":...",2025-11-03 17:19:51,2025-11-03,2025-11,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
1,3605520,02176d9a-3d7e-4461-86f7-0292bf3f1098,60836055200018,alpha_stack_model_sil,0.0927,"{""sb_demo_score"": 0.0420755117, ""s_cic_score"":...",2025-08-07 16:11:43,2025-08-07,2025-08,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
2,3675949,02772349-b50e-4d16-832e-0365c0365562,60836759490018,alpha_stack_model_sil,0.1844,"{""sb_demo_score"": 0.1058501955, ""s_cic_score"":...",2025-09-10 13:59:11,2025-09-10,2025-09,Prod,0,1,SIL Competitor,v1,Trench 2,Mall,android,NA,NA
3,2744135,02e4edb7-a5de-4415-a034-07b63370b61b,60827441350026,alpha_stack_model_sil,0.0720,"{""sb_demo_score"": 0.0785887604, ""s_cic_score"":...",2025-05-01 18:50:22,2025-05-01,2025-05,Prod,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
4,3708286,0467914c-ff76-44d0-8c80-aa4adac062b7,60837082860015,alpha_stack_model_sil,0.1102,"{""sb_demo_score"": 0.063306459, ""s_cic_score"": ...",2025-09-27 14:35:27,2025-09-27,2025-09,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA


In [6]:
df1 = dfd.copy()

### Train 

In [7]:
sq = """
with modelname as 
  (  SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
  case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    Data_selection,
       deviceOs osType,
    FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
  -- and modelVersionId = 'v1'
      ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  coalesce(cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.credo_score')AS FLOAT64), 
  cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.s_credo_score')AS FLOAT64)) AS credo_score,
  calcFeature,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  Data_selection,  
  deffpd0,
  flg_mature_fpd0,
  loanmaster.new_loan_type,
  modelVersionId, r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
        coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
                coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  left join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
 left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where 
  loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and flg_mature_fpd0 = 1
  )
  select *
  from base
  where credo_score is not null
 qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
   ;
  """
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()

The shape of the dataframe downloaded is:	 (296285, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,credo_score,calcFeature,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd0,flg_mature_fpd0,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,2395576,01709ba2-60f3-4ebe-9989-49b20a9b330e,60823955760014,Alpha - StackingModel,0.051384,"{""sb_demo_score"": 0.06381763591913761, ""s_cic_...",2024-02-10 10:30:01,2024-02-10,2024-02,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
1,2280491,0278cb4e-dcfc-475e-a9b5-d60cce5777cd,60822804910019,Alpha - StackingModel,0.097435,"{""sb_demo_score"": 0.05838055121806981, ""s_cic_...",2023-10-22 10:19:57,2023-10-22,2023-10,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
2,2283730,03ed7615-a5c3-4902-9c63-6076c8d8ce96,60822837300017,Alpha - StackingModel,0.133606,"{""sb_demo_score"": 0.09381565019044892, ""s_cic_...",2023-10-25 10:42:47,2023-10-25,2023-10,Pre_Train,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
3,2005936,0a545052-8a82-4d89-a9a3-74422cab0483,60820059360015,Alpha - StackingModel,0.042014,"{""sb_demo_score"": 0.06997884488860857, ""s_cic_...",2023-04-20 18:55:27,2023-04-20,2023-04,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
4,2210832,0d399430-96f1-4fa7-a9a8-f6c2990b6bb3,60822108320015,Alpha - StackingModel,0.110763,"{""sb_demo_score"": 0.16036951145526607, ""s_cic_...",2023-08-28 10:55:50,2023-08-29,2023-08,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA


In [8]:
df2 = dfd.copy()

### Concat 

In [9]:
# df_concat = pd.concat([df1, df2], ignore_index=True)
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")

# df_concat = (df_concat
#              .sort_values(by=['digitalLoanAccountId', 'Data_selection'],
#                           key=lambda s: s.map({'Train': 0, 'Test': 1}))
#              .drop_duplicates(subset=['digitalLoanAccountId'], keep='first')
#              .reset_index(drop=True))
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# df_concat.head()


# 1) Get all IDs present in Train
train_ids = set(df2['digitalLoanAccountId'])

# 2) Keep only Test rows whose ID is NOT in Train
df1_no_dupes = df1[~df1['digitalLoanAccountId'].isin(train_ids)]

# 3) Concatenate
df_concat = pd.concat([df1_no_dupes, df2], ignore_index=True)

print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
df_concat.head()


The shape of the concatenated dataframe is:	 (343756, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,credo_score,calcFeature,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd0,flg_mature_fpd0,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3789565,014b2b18-a79c-4083-937a-8a2e123c3068,60837895650013,alpha_stack_model_sil,0.1300,"{""sb_demo_score"": 0.0386566044, ""s_cic_score"":...",2025-11-03 17:19:51,2025-11-03,2025-11,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
1,2771736,06b38cea-aa76-4643-a241-a05b364a2058,60827717360011,alpha_stack_model_sil,0.0744,"{""sb_demo_score"": 0.1485844343, ""s_cic_score"":...",2025-10-28 17:56:58,2025-10-30,2025-10,Prod,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
2,3836654,07472f85-688a-48b2-af19-ae98594a3754,60838366540012,alpha_stack_model_sil,0.0927,"{""sb_demo_score"": 0.1498474067, ""s_cic_score"":...",2025-11-26 16:43:01,2025-11-26,2025-11,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
3,3752820,0a40df14-f644-4c21-a1cb-fc2baeeadf27,60837528200016,alpha_stack_model_sil,0.1217,"{""sb_demo_score"": 0.0208573521, ""s_cic_score"":...",2025-10-18 18:01:44,2025-10-18,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Mall,android,NA,NA
4,3780305,10e21741-56b7-4c76-a888-ac285ce76679,60837803050019,alpha_stack_model_sil,0.1439,"{""sb_demo_score"": 0.0525867868, ""s_cic_score"":...",2025-10-30 18:13:58,2025-10-30,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA


In [10]:
# df_concat = df1.copy()

df_concat['credo_score'] = pd.to_numeric(df_concat['credo_score'], errors='coerce')

In [11]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
    df_concat, 
    'credo_score', 
    'deffpd0', 
    'FPD0',
    data_selection_column='Data_selection',
    model_version_column='modelVersionId',
    trench_column='trenchCategory',
    loan_type_column='new_loan_type',
    loan_product_type_column='loan_product_type',
    ostype_column='osType', 
    risk_segment_column='risk_segment',
    risk_segment_final_column='risk_segment_final',
    account_id_column='digitalLoanAccountId'
)

In [13]:
fact_table, dimension_table = update_tables(fact_table, dimension_table, model_name='alpha_stack_credo_score_sil', product='SIL')
print(f"The shape of the fact table is:\t {fact_table.shape}")
print(f"The shape of the dimension table is:\t {dimension_table.shape}")

The shape of the fact table is:	 (326608, 19)
The shape of the dimension table is:	 (7784, 14)


In [14]:
facttable_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table_sil_1"
dimtable_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table_sil_1"

In [15]:
# Upload to BigQuery

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(fact_table, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=a7b7ce83-1da1-4ec7-ad18-f49c42823383>

In [16]:
# Upload to BigQuery

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dimension_table, dimtable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=a47341f7-8f55-40dc-8aa6-ff4a5b67d499>

## FPD10 

## Test 

In [17]:
sq = f"""
with modelname as 
  (  SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
  case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
      deviceOs osType,
    FROM prj-prod-dataplatform.audit_balance.ml_model_run_details mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
  -- and modelVersionId = 'v1'
      ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where  modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  coalesce(cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.credo_score')AS FLOAT64), 
  cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.s_credo_score')AS FLOAT64)) AS credo_score,
  calcFeature,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  'Prod' Data_selection,  
  del.deffpd10,
  flg_mature_fpd10,
  loanmaster.new_loan_type,
  modelVersionId, r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
        coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
     coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  left join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
  left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where 
  loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and flg_mature_fpd10 = 1
  )
  select *
  from base
  where credo_score is not null
 qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
   ;
"""
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()


The shape of the dataframe downloaded is:	 (78917, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,credo_score,calcFeature,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd10,flg_mature_fpd10,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3659016,015f7acd-1dd5-4c2a-906e-ca738a4f3589,60836590160018,alpha_stack_model_sil,0.1084,"{""sb_demo_score"": 0.0632928074, ""s_cic_score"":...",2025-09-02 12:11:04,2025-09-02,2025-09,Prod,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
1,3544893,01c48ce7-5eb8-43dc-ad84-466afdbb18f7,60835448930014,alpha_stack_model_sil,0.1997,"{""sb_demo_score"": 0.0907005225, ""s_cic_score"":...",2025-07-08 11:06:30,2025-07-08,2025-07,Prod,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
2,3646350,02241ff2-5018-4dc9-ace6-ba0abe556f20,60836463500012,alpha_stack_model_sil,0.0920,"{""sb_demo_score"": 0.0476905131, ""s_cic_score"":...",2025-08-27 14:59:49,2025-08-27,2025-08,Prod,0,1,SIL-Instore,v1,Trench 3,Appliance,ios,NA,NA
3,3775761,02aa9b40-7dc4-4080-8634-55a7b53a30d6,60837757610014,alpha_stack_model_sil,0.0965,"{""sb_demo_score"": 0.0386170282, ""s_cic_score"":...",2025-10-28 16:23:31,2025-10-28,2025-10,Prod,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
4,3750527,031506b2-8051-4cd9-a5e1-3e45817e8454,60837505270016,alpha_stack_model_sil,0.0591,"{""sb_demo_score"": 0.080426185, ""s_cic_score"": ...",2025-10-17 16:43:23,2025-10-17,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA


In [18]:
df1 = dfd.copy()

### Train 

In [19]:
sq = """
with modelname as 
  (  SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
  case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    Data_selection,
       deviceOs osType,
    FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
  -- and modelVersionId = 'v1'
      ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  coalesce(cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.credo_score')AS FLOAT64), 
  cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.s_credo_score')AS FLOAT64)) AS credo_score,
  calcFeature,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  Data_selection,  
  del.deffpd10,
  flg_mature_fpd10,
  loanmaster.new_loan_type,
  modelVersionId, r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
        coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
                coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  left join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
 left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where 
  loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and flg_mature_fpd10 = 1
  )
  select *
  from base
  where credo_score is not null
 qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
   ;
  """
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()

The shape of the dataframe downloaded is:	 (296285, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,credo_score,calcFeature,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd10,flg_mature_fpd10,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,1601750,05cdf3d8-32c6-479e-bf79-f828de25fa25,60816017500028,Alpha - StackingModel,0.185331,"{""sb_demo_score"": 0.14354616408514106, ""s_cic_...",2024-02-26 13:21:41,2024-02-26,2024-02,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
1,2256718,083681aa-84df-44bf-8e1d-cb6f9ad310e4,60822567180018,Alpha - StackingModel,0.231422,"{""sb_demo_score"": 0.11319933898827776, ""s_cic_...",2023-10-02 14:41:41,2023-10-02,2023-10,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
2,2240964,11de18a5-ee80-49fb-9926-166635eaf849,60822409640011,Alpha - StackingModel,0.145280,"{""sb_demo_score"": 0.11501963072165183, ""s_cic_...",2023-09-20 13:32:20,2023-09-20,2023-09,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
3,2234588,1c1b0e25-3073-4f71-b801-5a7bf94e567f,60822345880018,Alpha - StackingModel,0.119615,"{""sb_demo_score"": 0.09128002273228863, ""s_cic_...",2023-09-15 15:37:02,2023-09-15,2023-09,Pre_Train,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
4,2051364,1d91dd58-1ff8-476b-8927-e2c5d514db06,60820513640011,Alpha - StackingModel,0.179984,"{""sb_demo_score"": 0.12046390674396802, ""s_cic_...",2023-05-19 10:43:25,2023-05-19,2023-05,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA


In [20]:
df2 = dfd.copy()

### Concat 

In [21]:
# df_concat = pd.concat([df1, df2], ignore_index=True)
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")

# df_concat = (df_concat
#              .sort_values(by=['digitalLoanAccountId', 'Data_selection'],
#                           key=lambda s: s.map({'Train': 0, 'Test': 1}))
#              .drop_duplicates(subset=['digitalLoanAccountId'], keep='first')
#              .reset_index(drop=True))
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# df_concat.head()


# 1) Get all IDs present in Train
train_ids = set(df2['digitalLoanAccountId'])

# 2) Keep only Test rows whose ID is NOT in Train
df1_no_dupes = df1[~df1['digitalLoanAccountId'].isin(train_ids)]

# 3) Concatenate
df_concat = pd.concat([df1_no_dupes, df2], ignore_index=True)

print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
df_concat.head()


The shape of the concatenated dataframe is:	 (334610, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,credo_score,calcFeature,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd10,flg_mature_fpd10,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3775761,02aa9b40-7dc4-4080-8634-55a7b53a30d6,60837757610014,alpha_stack_model_sil,0.0965,"{""sb_demo_score"": 0.0386170282, ""s_cic_score"":...",2025-10-28 16:23:31,2025-10-28,2025-10,Prod,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
1,3750527,031506b2-8051-4cd9-a5e1-3e45817e8454,60837505270016,alpha_stack_model_sil,0.0591,"{""sb_demo_score"": 0.080426185, ""s_cic_score"": ...",2025-10-17 16:43:23,2025-10-17,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
2,3788367,0362add4-045a-45b8-9bbd-90b6ff45e27b,60837883670015,alpha_stack_model_sil,0.1569,"{""sb_demo_score"": 0.0513273726, ""s_cic_score"":...",2025-11-03 10:23:11,2025-11-03,2025-11,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
3,3822564,09687d93-cd4c-4db3-afff-dd103cca4fcd,60838225640011,alpha_stack_model_sil,0.0567,"{""sb_demo_score"": 0.0418090329, ""s_cic_score"":...",2025-11-20 14:33:29,2025-11-20,2025-11,Prod,0,1,SIL-Instore,v1,Trench 2,Mall,android,NA,NA
4,3800930,0b6cde8b-8c61-41c6-a59c-285d02cbdadf,60838009300014,alpha_stack_model_sil,0.1071,"{""sb_demo_score"": 0.0417215319, ""s_cic_score"":...",2025-11-09 13:08:52,2025-11-09,2025-11,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,ios,NA,NA


In [22]:
# df_concat = df1.copy()

df_concat['credo_score'] = pd.to_numeric(df_concat['credo_score'], errors='coerce')

In [23]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
    df_concat, 
    'credo_score', 
    'deffpd10', 
    'FPD10',
    data_selection_column='Data_selection',
    model_version_column='modelVersionId',
    trench_column='trenchCategory',
    loan_type_column='new_loan_type',
    loan_product_type_column='loan_product_type',
        ostype_column='osType',
        risk_segment_column='risk_segment',
        risk_segment_final_column='risk_segment_final',
    account_id_column='digitalLoanAccountId'
)

In [24]:
fact_table, dimension_table = update_tables(fact_table, dimension_table, model_name='alpha_stack_credo_score_sil', product='SIL')
print(f"The shape of the fact table is:\t {fact_table.shape}")
print(f"The shape of the dimension table is:\t {dimension_table.shape}")

The shape of the fact table is:	 (320624, 19)
The shape of the dimension table is:	 (7784, 14)


In [25]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(fact_table, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=c475d594-ffce-43ea-b8a5-93a71bf2fdc9>

In [26]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dimension_table, dimtable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=7a29342b-a7b2-4941-bb21-03cc40302841>

## FPD30 

## Test 

In [27]:
sq = f"""
with modelname as 
  (  SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
  case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
      deviceOs osType,
    FROM prj-prod-dataplatform.audit_balance.ml_model_run_details mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
  -- and modelVersionId = 'v1'
      ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where  modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  coalesce(cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.credo_score')AS FLOAT64), 
  cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.s_credo_score')AS FLOAT64)) AS credo_score,
  calcFeature,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  'Prod' Data_selection,  
  del.deffpd30,
  flg_mature_fpd30,
  loanmaster.new_loan_type,
  modelVersionId, r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
        coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
     coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  left join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
  left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where 
  loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and flg_mature_fpd30 = 1
  )
  select *
  from base
  where credo_score is not null
 qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
   ;
"""
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()


The shape of the dataframe downloaded is:	 (63123, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,credo_score,calcFeature,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd30,flg_mature_fpd30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3649205,006780cb-15ce-4e2f-9283-700e71cb7803,60836492050016,alpha_stack_model_sil,0.0965,"{""sb_demo_score"": 0.1689091324, ""s_cic_score"":...",2025-08-29 09:43:03,2025-08-29,2025-08,Prod,0,1,SIL ZERO,v1,Trench 2,Appliance,android,NA,NA
1,3746448,00b47e04-3aad-46d7-88c6-ba12112341f4,60837464480017,alpha_stack_model_sil,0.0849,"{""sb_demo_score"": 0.1152298084, ""s_cic_score"":...",2025-10-15 17:25:04,2025-10-15,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
2,3190221,01a18923-5dd3-4f67-8bfe-827cfc9f0ecf,60831902210015,alpha_stack_model_sil,0.0864,"{""sb_demo_score"": 0.0957833094, ""s_cic_score"":...",2025-07-07 14:46:28,2025-07-07,2025-07,Prod,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
3,3643106,02221103-7df1-4d97-979c-d094638b3d55,60836431060011,alpha_stack_model_sil,0.0769,"{""sb_demo_score"": 0.042758571, ""s_cic_score"": ...",2025-08-25 16:10:09,2025-08-25,2025-08,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
4,3505134,0300923b-c977-432f-98c5-42baaed180c7,60835051340016,alpha_stack_model_sil,0.0602,"{""sb_demo_score"": 0.0903301274, ""s_cic_score"":...",2025-06-18 13:52:02,2025-06-18,2025-06,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA


In [28]:
df1 = dfd.copy()

### Train 

In [29]:
sq = """
with modelname as 
  (  SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
  case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    Data_selection,
       deviceOs osType,
    FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
  -- and modelVersionId = 'v1'
      ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  coalesce(cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.credo_score')AS FLOAT64), 
  cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.s_credo_score')AS FLOAT64)) AS credo_score,
  calcFeature,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  Data_selection,  
  del.deffpd30,
  flg_mature_fpd30,
  loanmaster.new_loan_type,
  modelVersionId, r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
        coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
                coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  left join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
 left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where 
  loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and flg_mature_fpd30 = 1
  )
  select *
  from base
  where credo_score is not null
 qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
   ;
  """
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()

The shape of the dataframe downloaded is:	 (296285, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,credo_score,calcFeature,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd30,flg_mature_fpd30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,2078571,01cd7b8f-a5f9-4741-8c1c-c01a47997734,60820785710013,Alpha - StackingModel,0.110134,"{""sb_demo_score"": 0.11520132203952498, ""s_cic_...",2023-06-05 12:18:25,2023-06-05,2023-06,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
1,2020778,01d62bb7-efbc-4a87-9a3c-ff3c2f65dbde,60820207780019,Alpha - StackingModel,0.077958,"{""sb_demo_score"": 0.08050267789755015, ""s_cic_...",2023-04-30 12:40:40,2023-04-30,2023-04,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
2,2261288,04c5150e-0456-4314-a428-270fce1efcf1,60822612880012,Alpha - StackingModel,0.109841,"{""sb_demo_score"": 0.0733760300146572, ""s_cic_s...",2023-10-05 19:04:56,2023-10-05,2023-10,Pre_Train,1,1,SIL-Instore,v1,Trench 2,Appliance,ios,NA,NA
3,2086602,0689b962-26b9-43b0-930f-7e1b9ede029c,60820866020011,Alpha - StackingModel,0.033723,"{""sb_demo_score"": 0.10291667270934084, ""s_cic_...",2023-06-10 16:27:21,2023-06-10,2023-06,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
4,2244363,0b8d931d-80b7-4361-b574-d7ecaf63139b,60822443630018,Alpha - StackingModel,0.147546,"{""sb_demo_score"": 0.10778766652379383, ""s_cic_...",2023-09-23 11:28:16,2023-09-23,2023-09,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA


In [30]:
df2 = dfd.copy()

### Concat 

In [31]:
# df_concat = pd.concat([df1, df2], ignore_index=True)
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")

# df_concat = (df_concat
#              .sort_values(by=['digitalLoanAccountId', 'Data_selection'],
#                           key=lambda s: s.map({'Train': 0, 'Test': 1}))
#              .drop_duplicates(subset=['digitalLoanAccountId'], keep='first')
#              .reset_index(drop=True))
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# df_concat.head()


# 1) Get all IDs present in Train
train_ids = set(df2['digitalLoanAccountId'])

# 2) Keep only Test rows whose ID is NOT in Train
df1_no_dupes = df1[~df1['digitalLoanAccountId'].isin(train_ids)]

# 3) Concatenate
df_concat = pd.concat([df1_no_dupes, df2], ignore_index=True)

print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
df_concat.head()


The shape of the concatenated dataframe is:	 (318816, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,credo_score,calcFeature,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffpd30,flg_mature_fpd30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3746448,00b47e04-3aad-46d7-88c6-ba12112341f4,60837464480017,alpha_stack_model_sil,0.0849,"{""sb_demo_score"": 0.1152298084, ""s_cic_score"":...",2025-10-15 17:25:04,2025-10-15,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
1,3756450,08934562-b8d6-4e0c-927d-c908c28d0cd2,60837564500016,alpha_stack_model_sil,0.0872,"{""sb_demo_score"": 0.0544801133, ""s_cic_score"":...",2025-10-20 13:53:20,2025-10-20,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
2,3587551,306cd44a-2c53-49f7-a233-64309f12fbf1,60835875510017,alpha_stack_model_sil,0.1480,"{""sb_demo_score"": 0.1591883109, ""s_cic_score"":...",2025-07-30 11:11:54,2025-08-05,2025-07,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
3,3763505,32083827-d6b9-43bd-a19e-25035c1f9051,60837635050017,alpha_stack_model_sil,0.2603,"{""sb_demo_score"": 0.0321400768, ""s_cic_score"":...",2025-10-23 17:16:30,2025-10-23,2025-10,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
4,3790998,39e200e9-a515-4364-b4c9-4f3fb0dbcdd5,60837909980013,alpha_stack_model_sil,0.1166,"{""sb_demo_score"": 0.0514740432, ""s_cic_score"":...",2025-11-04 14:36:25,2025-11-04,2025-11,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA


In [32]:
df_concat['credo_score'] = pd.to_numeric(df_concat['credo_score'], errors='coerce')

In [33]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
    df_concat, 
    'credo_score', 
    'deffpd30', 
    'FPD30',
    data_selection_column='Data_selection',
    model_version_column='modelVersionId',
    trench_column='trenchCategory',
    loan_type_column='new_loan_type',
    loan_product_type_column='loan_product_type',
      ostype_column='osType',  
        risk_segment_column='risk_segment',
            risk_segment_final_column='risk_segment_final',
    account_id_column='digitalLoanAccountId'
)

In [34]:
fact_table, dimension_table = update_tables(fact_table, dimension_table, model_name='alpha_stack_credo_score_sil', product='SIL')
print(f"The shape of the fact table is:\t {fact_table.shape}")
print(f"The shape of the dimension table is:\t {dimension_table.shape}")

The shape of the fact table is:	 (312608, 19)
The shape of the dimension table is:	 (7720, 14)


In [35]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(fact_table, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=a09f1751-31d0-4a93-8317-7d498dabdd13>

In [36]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dimension_table, dimtable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=76248a12-1646-4238-a0f8-0842166bcdba>

## FSPD30 

## Test 

In [37]:
sq = f"""
with modelname as 
  (  SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
  case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
      deviceOs osType,
    FROM prj-prod-dataplatform.audit_balance.ml_model_run_details mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
  -- and modelVersionId = 'v1'
      ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where  modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  coalesce(cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.credo_score')AS FLOAT64), 
  cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.s_credo_score')AS FLOAT64)) AS credo_score,
  calcFeature,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  'Prod' Data_selection,  
  del.deffspd30,
  flg_mature_fspd_30,
  loanmaster.new_loan_type,
  modelVersionId, r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
        coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
     coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  left join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
  left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where 
  loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and flg_mature_fspd_30 = 1
  )
  select *
  from base
  where credo_score is not null
 qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
   ;
"""
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()


The shape of the dataframe downloaded is:	 (48616, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,credo_score,calcFeature,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffspd30,flg_mature_fspd_30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3572406,002fa325-8e68-4efc-9238-cdeeb58f0cd7,60835724060014,alpha_stack_model_sil,0.1179,"{""sb_demo_score"": 0.0650061873, ""s_cic_score"":...",2025-07-22 17:59:56,2025-07-22,2025-07,Prod,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
1,3712616,05f0fcd7-c3be-49ac-afd9-9e26f8bced24,60837126160014,alpha_stack_model_sil,0.1810,"{""sb_demo_score"": 0.0418868898, ""s_cic_score"":...",2025-09-29 13:33:01,2025-09-29,2025-09,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,android,NA,NA
2,3703769,08c7c322-35da-4472-b712-1e9bbfe33917,60837037690019,alpha_stack_model_sil,0.1729,"{""sb_demo_score"": 0.1038476227, ""s_cic_score"":...",2025-09-25 08:52:27,2025-09-25,2025-09,Prod,0,1,SIL Competitor,v1,Trench 2,Appliance,ios,NA,NA
3,3469691,0cb97fa9-8568-4eb2-a0d2-74840fdee48d,60834696910011,alpha_stack_model_sil,0.0878,"{""sb_demo_score"": 0.0261010805, ""s_cic_score"":...",2025-05-31 15:11:26,2025-05-31,2025-05,Prod,0,1,SIL-Instore,v1,Trench 2,Mall,android,NA,NA
4,3494199,0d2547e3-6c11-495f-a816-e5d7235b8965,60834941990018,alpha_stack_model_sil,0.1147,"{""sb_demo_score"": 0.0357694842, ""s_cic_score"":...",2025-06-13 10:15:04,2025-06-13,2025-06,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,ios,NA,NA


In [38]:
df1 = dfd.copy()

### Train 

In [39]:
sq = """
with modelname as 
  (  SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
  case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    Data_selection,
       deviceOs osType,
    FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
  -- and modelVersionId = 'v1'
      ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  coalesce(cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.credo_score')AS FLOAT64), 
  cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.s_credo_score')AS FLOAT64)) AS credo_score,
  calcFeature,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  Data_selection,  
  del.deffspd30,
  flg_mature_fspd_30,
  loanmaster.new_loan_type,
  modelVersionId, r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
        coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
                coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  left join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
 left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where 
  loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and flg_mature_fspd_30 = 1
  )
  select *
  from base
  where credo_score is not null
 qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
   ;
  """
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()

The shape of the dataframe downloaded is:	 (296285, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,credo_score,calcFeature,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffspd30,flg_mature_fspd_30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,2395576,01709ba2-60f3-4ebe-9989-49b20a9b330e,60823955760014,Alpha - StackingModel,0.051384,"{""sb_demo_score"": 0.06381763591913761, ""s_cic_...",2024-02-10 10:30:01,2024-02-10,2024-02,Pre_Train,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
1,2280491,0278cb4e-dcfc-475e-a9b5-d60cce5777cd,60822804910019,Alpha - StackingModel,0.097435,"{""sb_demo_score"": 0.05838055121806981, ""s_cic_...",2023-10-22 10:19:57,2023-10-22,2023-10,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
2,2283730,03ed7615-a5c3-4902-9c63-6076c8d8ce96,60822837300017,Alpha - StackingModel,0.133606,"{""sb_demo_score"": 0.09381565019044892, ""s_cic_...",2023-10-25 10:42:47,2023-10-25,2023-10,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
3,2005936,0a545052-8a82-4d89-a9a3-74422cab0483,60820059360015,Alpha - StackingModel,0.042014,"{""sb_demo_score"": 0.06997884488860857, ""s_cic_...",2023-04-20 18:55:27,2023-04-20,2023-04,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
4,2210832,0d399430-96f1-4fa7-a9a8-f6c2990b6bb3,60822108320015,Alpha - StackingModel,0.110763,"{""sb_demo_score"": 0.16036951145526607, ""s_cic_...",2023-08-28 10:55:50,2023-08-29,2023-08,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA


In [40]:
df2 = dfd.copy()

### Concat 

In [41]:
# df_concat = pd.concat([df1, df2], ignore_index=True)
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")

# df_concat = (df_concat
#              .sort_values(by=['digitalLoanAccountId', 'Data_selection'],
#                           key=lambda s: s.map({'Train': 0, 'Test': 1}))
#              .drop_duplicates(subset=['digitalLoanAccountId'], keep='first')
#              .reset_index(drop=True))
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# df_concat.head()


# 1) Get all IDs present in Train
train_ids = set(df2['digitalLoanAccountId'])

# 2) Keep only Test rows whose ID is NOT in Train
df1_no_dupes = df1[~df1['digitalLoanAccountId'].isin(train_ids)]

# 3) Concatenate
df_concat = pd.concat([df1_no_dupes, df2], ignore_index=True)

print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
df_concat.head()


The shape of the concatenated dataframe is:	 (304309, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,credo_score,calcFeature,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffspd30,flg_mature_fspd_30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3469691,0cb97fa9-8568-4eb2-a0d2-74840fdee48d,60834696910011,alpha_stack_model_sil,0.0878,"{""sb_demo_score"": 0.0261010805, ""s_cic_score"":...",2025-05-31 15:11:26,2025-05-31,2025-05,Prod,0,1,SIL-Instore,v1,Trench 2,Mall,android,NA,NA
1,3792721,5558aaf3-db5e-42bf-ae34-1a135dc8f2e5,60837927210014,alpha_stack_model_sil,0.0790,"{""sb_demo_score"": 0.0556081153, ""s_cic_score"":...",2025-11-05 13:21:41,2025-11-05,2025-11,Prod,0,1,SIL ZERO,v1,Trench 2,Appliance,android,NA,NA
2,3791353,c4acd610-127a-4700-a143-1c3384f24101,60837913530016,alpha_stack_model_sil,0.1514,"{""sb_demo_score"": 0.06927308, ""s_cic_score"": 0...",2025-11-04 16:34:19,2025-11-04,2025-11,Prod,0,1,SIL-Instore,v1,Trench 2,Mall,ios,NA,NA
3,3792371,e2d7d16f-6cb8-4d8c-8ca2-13411678fc58,60837923710012,alpha_stack_model_sil,0.1376,"{""sb_demo_score"": 0.039181198, ""s_cic_score"": ...",2025-11-05 10:27:29,2025-11-05,2025-11,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
4,3792583,e7e44b3a-175c-42ed-9a01-e4336c0e2ea0,60837925830012,alpha_stack_model_sil,0.1279,"{""sb_demo_score"": 0.055847233, ""s_cic_score"": ...",2025-11-05 12:17:39,2025-11-05,2025-11,Prod,0,1,SIL-Instore,v1,Trench 2,Mall,android,NA,NA


In [42]:
# df_concat = df1.copy()

df_concat['credo_score'] = pd.to_numeric(df_concat['credo_score'], errors='coerce')

In [43]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
    df_concat, 
    'credo_score', 
    'deffspd30', 
    'FSPD30',
    data_selection_column='Data_selection',
    model_version_column='modelVersionId',
    trench_column='trenchCategory',
    loan_type_column='new_loan_type',
    loan_product_type_column='loan_product_type',
    ostype_column='osType', 
    risk_segment_column='risk_segment',
    risk_segment_final_column='risk_segment_final',
    account_id_column='digitalLoanAccountId'
)

In [44]:
fact_table, dimension_table = update_tables(fact_table, dimension_table, model_name='alpha_stack_credo_score_sil', product='SIL')
print(f"The shape of the fact table is:\t {fact_table.shape}")
print(f"The shape of the dimension table is:\t {dimension_table.shape}")

The shape of the fact table is:	 (300160, 19)
The shape of the dimension table is:	 (7124, 14)


In [45]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(fact_table, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=ccf68543-be14-44f8-9e41-eeb0edb7058d>

In [46]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dimension_table, dimtable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=c943771a-1137-419d-8083-2ccf14fce0e6>

## FSTPD30 

## Test 

In [47]:
sq = f"""with modelname as 
  (  SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
  case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
      deviceOs osType,
    FROM prj-prod-dataplatform.audit_balance.ml_model_run_details mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
  -- and modelVersionId = 'v1'
      ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where  modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  coalesce(cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.credo_score')AS FLOAT64), 
  cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.s_credo_score')AS FLOAT64)) AS credo_score,
  calcFeature,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  'Prod' Data_selection,  
  del.deffstpd30,
  flg_mature_fstpd_30,
  loanmaster.new_loan_type,
  modelVersionId, r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
        coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
     coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  left join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
  left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where 
  loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and flg_mature_fstpd_30 = 1
  )
  select *
  from base
  where credo_score is not null
 qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
   ;
"""
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()


The shape of the dataframe downloaded is:	 (37538, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,credo_score,calcFeature,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffstpd30,flg_mature_fstpd_30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3471653,0becf72a-0667-4b49-9333-58e45ef000ca,60834716530017,alpha_stack_model_sil,0.1071,"{""sb_demo_score"": 0.0451478456, ""s_cic_score"":...",2025-06-01 13:56:49,2025-06-01,2025-06,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,ios,NA,NA
1,3720614,125c3b59-ec7f-4488-93d3-0dcf62dcebf8,60837206140019,alpha_stack_model_sil,0.0996,"{""sb_demo_score"": 0.0561239103, ""s_cic_score"":...",2025-10-03 14:24:20,2025-10-03,2025-10,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
2,3490923,142b6f50-cf87-4c3c-a402-53d36bd94b22,60834909230016,alpha_stack_model_sil,0.0838,"{""sb_demo_score"": 0.0216179348, ""s_cic_score"":...",2025-06-11 14:26:13,2025-06-11,2025-06,Prod,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
3,3478895,16e9ebc1-d620-45ee-ab7b-29b42c37bf75,60834788950011,alpha_stack_model_sil,0.1688,"{""sb_demo_score"": 0.0413912442, ""s_cic_score"":...",2025-06-05 11:29:22,2025-06-05,2025-06,Prod,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
4,2305595,3c2d3de7-b135-438a-9e54-7b2e8be0738b,60823055950021,alpha_stack_model_sil,0.1514,"{""sb_demo_score"": 0.1344546945, ""s_cic_score"":...",2025-06-03 14:13:33,2025-06-03,2025-06,Prod,0,1,SIL-Instore,v1,Trench 3,Appliance,ios,NA,NA


In [48]:
df1 = dfd.copy()

### Train 

In [49]:
sq = """
with modelname as 
  (  SELECT
    mmrd.customerId,mmrd.digitalLoanAccountId,prediction,start_time,end_time,modelDisplayName,modelVersionId, 
  case when trenchCategory is null then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench1' end) 
     when trenchCategory = '' then (case when mt.ln_user_type='1_Repeat Applicant' then 'Trench 3'
    when mt.ln_user_type <>'1_Repeat Applicant' and DATE_DIFF(current_date(), mt.onb_tsa_onboarding_datetime, DAY) >30 then 'Trench 2'
    else 'Trench 1' end) 
    else trenchCategory end  as trenchCategory,
    REPLACE(REPLACE(calcFeature, "'", '"'), "None", "null") AS calcFeature,
    Data_selection,
       deviceOs osType,
    FROM prj-prod-dataplatform.dap_ds_poweruser_playground.ml_training_model_run_details_20260116 mmrd
  left join prj-prod-dataplatform.risk_credit_mis.model_loan_score_mart mt on mt.digitalLoanAccountId = mmrd.digitalLoanAccountId
  WHERE modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
  -- and modelVersionId = 'v1'
      ),
  deliquency as
(select loanAccountNumber,
case when obs_min_inst_def0 >= 1 and min_inst_def0 = 1 then 1 else 0 end deffpd0,
case when obs_min_inst_def10 >=1 and min_inst_def10 =1 then 1 else 0 end deffpd10,
case when obs_min_inst_def30 >=1 and min_inst_def30 =1 then 1 else 0 end deffpd30,
case when obs_min_inst_def30 >=2 and min_inst_def30 in (1,2) then 1 else 0 end deffspd30,
case when obs_min_inst_def30 >=3 and min_inst_def30 in (1,2,3) then 1 else 0 end deffstpd30,
case when obs_min_inst_def0 >= 1 then 1 else 0 end flg_mature_fpd0,
case when obs_min_inst_def10 >=1 then 1 else 0 end flg_mature_fpd10,
case when obs_min_inst_def30 >=1 then 1 else 0 end flg_mature_fpd30,
case when obs_min_inst_def30 >=2 then 1 else 0 end flg_mature_fspd_30,
case when obs_min_inst_def30 >=3 then 1 else 0 end flg_mature_fstpd_30
from prj-prod-dataplatform.risk_credit_mis.loan_deliquency_data),
segmentdata AS (
SELECT loan.customerid , loan.digitalLoanAccountId, trench_category.trenchCategory,
loan.offer_id,
CASE
WHEN COALESCE(trench1_seg.risk_segment) IS NULL
THEN 'Unsegmented'
ELSE COALESCE(trench1_seg.risk_segment)
END AS risk_segment,
appVersion,flagApproval,tsa_onboarding_time,
IF(applicationStatus  in ('COMPLETED','ACTIVATED','APPROVED') , 'Loan Approved', 'Loan Not Approved') AS loan_application_status,
--if(disbursementDateTime is not null, 'Loan Disbursed', 'Loan Not Approved') loan_application_status
DATE(decision_date) AS application_date  
from
(select distinct digitalLoanAccountId, customerId,applicationStatus,disbursementDateTime,date(decision_date) decision_date, appVersion,flagApproval,tsa_onboarding_time,offer_id from `risk_credit_mis.loan_master_table`
  where date(decision_date) >=date('2025-11-10') and new_loan_type='Quick'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY customerId ORDER BY decision_date desc)=1
) loan
left join (SELECT digitalLoanAccountId,case when trenchCategory='Trench 1' then 'Trench-1'  when trenchCategory='Trench 2' then 'Trench-2'
when trenchCategory='Trench 3' then 'Trench-3' end as trenchCategory
,publish_time FROM `audit_balance.ml_model_run_details`
where modelDisplayName in  ('Alpha - StackingModel', 'alpha_stack_model_sil')
qualify row_number() over(partition by digitalLoanAccountId order by publish_time desc)=1
) trench_category on trench_category.digitalLoanAccountId=loan.digitalLoanAccountId
LEFT JOIN (
SELECT
cust_id,risk_segment,created_date,created_by,offer_id FROM `dl_loans_db_raw.tdbk_loan_offers_trx`
WHERE offer_type='SEGMENTED_ACL'
--AND created_by='GCP-API-CALL'
--QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_id ORDER BY created_date desc)=1
) trench1_seg ON trench1_seg.offer_id=loan.offer_id
),
base as 
  (select distinct r.customerId,
  r.digitalLoanAccountId,
  loanmaster.loanAccountNumber,
  r.modelDisplayName,
  coalesce(cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.credo_score')AS FLOAT64), 
  cast(JSON_VALUE(SAFE.PARSE_JSON(CAST(calcFeature AS STRING)), '$.s_credo_score')AS FLOAT64)) AS credo_score,
  calcFeature,
  coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime)) AS appln_submit_datetime,
  date(loanmaster.disbursementDateTime) disbursementdate,
  format_date('%Y-%m', coalesce(IF(loanmaster.new_loan_type = 'Flex-up', loanmaster.startApplyDateTime, loanmaster.termsAndConditionsSubmitDateTime),  cast(r.start_time as datetime))) as Application_month, 
  Data_selection,  
  del.deffstpd30,
  flg_mature_fstpd_30,
  loanmaster.new_loan_type,
  modelVersionId, r.trenchCategory,
    case when loanmaster.loantype='BNPL' and store_type =1 then 'Appliance'
    when loanmaster.loantype='BNPL' and store_type =2 then 'Mobile' 
    when loanmaster.loantype='BNPL' and store_type =3 then 'Mall' 
    when loanmaster.loantype='BNPL' and store_type not in (1,2,3) then store_tagging
    else 'not applicable' end as loan_product_type,
        coalesce((case when lower(r.osType) like '%andro%' then 'android'
                  when lower(r.osType) like '%os%' then 'ios' else lower(r.osType) end), 
            (case when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%andro%' then 'android'
                  when lower(coalesce(loanmaster.osversion_v2, loanmaster.osVersion)) like '%os%' then 'ios'
                  when lower(loanmaster.deviceType) like '%andro%' then 'android'
                  else 'ios' end)
            ) as osType,
                coalesce(sd.risk_segment, 'NA') risk_segment,
  coalesce(frs.risk_segment_final, 'NA')risk_segment_final
  from modelname r
  left join risk_credit_mis.loan_master_table loanmaster  ON loanmaster.digitalLoanAccountId = r.digitalLoanAccountId
  left join deliquency del on del.loanAccountNumber = loanmaster.loanAccountNumber
   left join(SELECT DISTINCT mer_refferal_code, mer_name mer_name,store_type,store_tagging FROM `dl_loans_db_raw.tdbk_merchant_refferal_mtb`
  left join worktable_datachampions.TARGET_SPLIT P on P.STORE_NAME = mer_name
 qualify row_number() over(partition by mer_refferal_code order by  created_dt desc)=1) sil_category on loanmaster.purpleKey=sil_category.mer_refferal_code
 left join segmentdata sd on sd.digitalLoanAccountId = loanmaster.digitalLoanAccountId
LEFT JOIN (
  select digitalLoanAccountid, risk_segment_final from prj-prod-dataplatform.dl_loans_db_raw.tdbk_loan_poi3_response where risk_segment_final is not null
qualify row_number() over(partition by digitalLoanAccountid order by created_dt desc) = 1
          ) frs on frs.digitalLoanAccountId = loanmaster.digitalLoanAccountId
  where 
  loanmaster.flagDisbursement = 1
  and loanmaster.disbursementDateTime is not null
  and flg_mature_fstpd_30 = 1
  )
  select *
  from base
  where credo_score is not null
 qualify row_number() over(partition by digitalLoanAccountId, modelVersionId order by appln_submit_datetime) = 1
   ;
  """
dfd = client.query(sq).to_dataframe()
# dfd = dfd.drop_duplicates(keep='first')
print(f"The shape of the dataframe downloaded is:\t {dfd.shape}")
dfd.head()

The shape of the dataframe downloaded is:	 (292421, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,credo_score,calcFeature,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffstpd30,flg_mature_fstpd_30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,2413987,001e561c-ba8e-4c59-817c-fe741f2b2955,60824139870017,Alpha - StackingModel,0.069076,"{""sb_demo_score"": 0.10182213445751781, ""s_cic_...",2024-02-28 14:55:02,2024-02-28,2024-02,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
1,2284657,005f6e89-402d-4aee-aebf-6c3b9fa6868d,60822846570019,Alpha - StackingModel,0.082544,"{""sb_demo_score"": 0.13025983473442665, ""s_cic_...",2023-10-26 10:29:20,2023-10-26,2023-10,Pre_Train,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
2,2261939,01989f73-836c-4fac-90e3-134021d8ce85,60822619390014,Alpha - StackingModel,0.219233,"{""sb_demo_score"": 0.06292621865787021, ""s_cic_...",2023-10-06 10:11:53,2023-10-06,2023-10,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA
3,2089245,06027c60-c5d2-442a-8be4-dc4c1ceba9b6,60820892450011,Alpha - StackingModel,0.064874,"{""sb_demo_score"": 0.06428366232210175, ""s_cic_...",2023-06-12 14:35:14,2023-06-12,2023-06,Pre_Train,0,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
4,2191071,076f6c2c-a6a5-48e7-98a6-8582ea00aa4d,60821910710016,Alpha - StackingModel,0.073883,"{""sb_demo_score"": 0.10841569746991231, ""s_cic_...",2023-08-15 18:58:05,2023-08-15,2023-08,Pre_Train,0,1,SIL-Instore,v1,Trench 3,Appliance,android,NA,NA


In [50]:
df2 = dfd.copy()

### Concat 

In [51]:
# df_concat = pd.concat([df1, df2], ignore_index=True)
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")

# df_concat = (df_concat
#              .sort_values(by=['digitalLoanAccountId', 'Data_selection'],
#                           key=lambda s: s.map({'Train': 0, 'Test': 1}))
#              .drop_duplicates(subset=['digitalLoanAccountId'], keep='first')
#              .reset_index(drop=True))
# print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
# df_concat.head()


# 1) Get all IDs present in Train
train_ids = set(df2['digitalLoanAccountId'])

# 2) Keep only Test rows whose ID is NOT in Train
df1_no_dupes = df1[~df1['digitalLoanAccountId'].isin(train_ids)]

# 3) Concatenate
df_concat = pd.concat([df1_no_dupes, df2], ignore_index=True)

print(f"The shape of the concatenated dataframe is:\t {df_concat.shape}")
df_concat.head()


The shape of the concatenated dataframe is:	 (292564, 19)


,customerId,digitalLoanAccountId,loanAccountNumber,modelDisplayName,credo_score,calcFeature,appln_submit_datetime,disbursementdate,Application_month,Data_selection,deffstpd30,flg_mature_fstpd_30,new_loan_type,modelVersionId,trenchCategory,loan_product_type,osType,risk_segment,risk_segment_final
0,3469691,0cb97fa9-8568-4eb2-a0d2-74840fdee48d,60834696910011,alpha_stack_model_sil,0.0878,"{""sb_demo_score"": 0.0261010805, ""s_cic_score"":...",2025-05-31 15:11:26,2025-05-31,2025-05,Prod,0,1,SIL-Instore,v1,Trench 2,Mall,android,NA,NA
1,3431236,3b19bc86-07f3-4f35-839d-ebd5c9c9f7f0,60834312360014,alpha_stack_model_sil,0.1564,"{""sb_demo_score"": 0.1476247707, ""s_cic_score"":...",2025-05-11 13:56:41,2025-05-11,2025-05,Prod,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
2,3714437,6d6b55e1-08ad-47ea-927e-ac2960914893,60837144370011,alpha_stack_model_sil,0.0951,"{""sb_demo_score"": 0.024321341, ""s_cic_score"": ...",2025-10-03 16:54:12,2025-10-03,2025-10,Prod,0,1,SIL Competitor,v1,Trench 2,Mall,android,NA,NA
3,3482558,e875982d-ea4b-492e-a5ef-a5304b679d4e,60834825580018,alpha_stack_model_sil,0.0964,"{""sb_demo_score"": 0.1491641468, ""s_cic_score"":...",2025-06-07 09:20:45,2025-06-11,2025-06,Prod,1,1,SIL-Instore,v1,Trench 2,Appliance,android,NA,NA
4,3501958,bd01f19a-682e-4324-a16b-a52b77a70dec,60835019580012,alpha_stack_model_sil,0.0719,"{""sb_demo_score"": 0.0246837221, ""s_cic_score"":...",2025-06-16 19:59:41,2025-06-16,2025-06,Prod,0,1,SIL Competitor,v1,Trench 2,Mall,android,NA,NA


In [52]:
# df_concat = df1.copy()

df_concat['credo_score'] = pd.to_numeric(df_concat['credo_score'], errors='coerce')

In [53]:
fact_table, dimension_table = calculate_periodic_gini_prod_ver_trench_dimfact(
    df_concat, 
    'credo_score', 
    'deffstpd30', 
    'FSTPD30',
    data_selection_column='Data_selection',
    model_version_column='modelVersionId',
    trench_column='trenchCategory',
    loan_type_column='new_loan_type',
    loan_product_type_column='loan_product_type',
     ostype_column='osType', 
    risk_segment_column='risk_segment',
    risk_segment_final_column='risk_segment_final',
    account_id_column='digitalLoanAccountId'
)

In [54]:
fact_table, dimension_table = update_tables(fact_table, dimension_table, model_name='alpha_stack_credo_score_sil', product='SIL')
print(f"The shape of the fact table is:\t {fact_table.shape}")
print(f"The shape of the dimension table is:\t {dimension_table.shape}")

The shape of the fact table is:	 (288776, 19)
The shape of the dimension table is:	 (6956, 14)


In [55]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.fact_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(fact_table, facttable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=256e90f0-b0d4-469e-b010-7d80486ca0ea>

In [ ]:
# Upload to BigQuery
# table_id = "prj-prod-dataplatform.dap_ds_poweruser_playground.dimension_table3"
job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_APPEND",  # or "WRITE_APPEND"
)
job = client.load_table_from_dataframe(dimension_table, dimtable_id, job_config=job_config)
job.result()  # Wait for the job to complete

C:\Users\Dwaipayan\AppData\Roaming\Python\Python312\site-packages\google\cloud\bigquery\_pandas_helpers.py:483: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


LoadJob<project=prj-prod-dataplatform, location=asia-southeast1, id=8d3bf692-c415-479d-8a8e-441786b78f22>

: 